In [ ]:
import face_recognition
import os
import cv2
import mysql.connector
from datetime import datetime
import smtplib
import time
import threading

known_faces_dir1 = "missingpeopleimg"
known_faces_dir2 = "criminalpeopleimg"

MODEL = "hog"
TOLERANCE = 0.5
DETECT_EVERY_N_FRAMES = 10
location = "Lab1"

known_faces = []
known_names = []

class VideoStream:
    def __init__(self, src=0, width=640, height=480):
        self.cap = cv2.VideoCapture(src)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
        self.ret, self.frame = self.cap.read()
        self.stopped = False
        threading.Thread(target=self.update, daemon=True).start()

    def update(self):
        while not self.stopped:
            self.ret, self.frame = self.cap.read()

    def read(self):
        return self.ret, self.frame

    def stop(self):
        self.stopped = True
        self.cap.release()

def updateCrowdStatus(people_count, crowd_level):
    try:
        conn = mysql.connector.connect(
            user='root',
            password='',
            host='127.0.0.1',
            database='kumbharakshak_db'
        )
        cursor = conn.cursor()
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        sql = """
        INSERT INTO crowd_status (location, people_count, density, last_updated)
        VALUES (%s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            people_count = VALUES(people_count),
            density = VALUES(density),
            last_updated = VALUES(last_updated)
        """
        
        cursor.execute(sql, (location, people_count, crowd_level, now))
        conn.commit()
        print("[INFO] Crowd status UPDATED - People: {}, Level: {}".format(people_count, crowd_level))

    except Exception as e:
        print("[ERROR] Crowd DB:", str(e))
    finally:
        try:
            cursor.close()
            conn.close()
        except:
            pass


def dataAdd(fname):
    try:
        conn = mysql.connector.connect(
            user='root',
            password='',
            host='127.0.0.1',
            database='kumbharakshak_db'
        )
        cursor = conn.cursor()
        now = datetime.now()

        sql = """INSERT INTO findlog(name, location, typee, datee, timee)
                 VALUES (%s, %s, %s, %s, %s)"""

        cursor.execute(sql, (
            fname, location, "Detected",
            now.strftime("%Y-%m-%d"),
            now.strftime("%H:%M:%S")
        ))
        conn.commit()
        conn.close()
    except:
        pass
    
def hasNewFaces():
    try:
        conn = mysql.connector.connect(
            user='root', password='', host='127.0.0.1', database='kumbharakshak_db'
        )
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM missingpeople")
        count = cursor.fetchone()[0]
        conn.close()
        return count != len(known_names)  # Reload if count differs
    except:
        return False
    
def sendEmail(fname):
    fromMy = 'harshwardhanmalode798@gmail.com'
    subj = 'Detection of Person'
    now = datetime.now()
    datee = now.strftime("%d/%m/%Y")
    timee = now.strftime("%H:%M:%S")
    message_text = fname + " Detected at " + location + " Date : " + datee + " Time : " + timee

    police_email = None
    added_by_email = None

    try:
        conn = mysql.connector.connect(
            user='root',
            password='',
            host='127.0.0.1',
            database='kumbharakshak_db'
        )
        cursor = conn.cursor()

        cursor.execute(
            "SELECT email FROM police_helpline WHERE location = %s LIMIT 1",
            (location,)
        )
        result = cursor.fetchone()
        if result:
            police_email = result[0]
        print(fname)
        cursor.execute(
            "SELECT added_by_email FROM missingpeople WHERE CONCAT(First_Name, ' ', Last_Name) = %s LIMIT 1",
            (fname,)
        )

        result = cursor.fetchone()
        if result:
            added_by_email = result[0]

        conn.close()

        if not police_email and not added_by_email:
            print("[ERROR] No recipient email found")
            return

    except Exception as e:
        print("[ERROR] DB error while fetching emails:", e)
        return

    recipients = []
    if police_email:
        recipients.append(police_email)
    if added_by_email:
        recipients.append(added_by_email)

    msg = "From: %s\nTo: %s\nSubject: %s\nDate: %s\n\n%s" % (
        fromMy,
        ", ".join(recipients),
        subj,
        datee,
        message_text
    )

    username = 'harshwardhanmalode798@gmail.com'
    password = 'mvqx endy emsj voys'

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.ehlo()
        server.starttls()
        server.login(username, password)
        server.sendmail(fromMy, recipients, msg)
        server.quit()
        print('ok the email has sent')
    except Exception as e:
        print("can't send the Email:", e)


def load_faces(directory):
    for name in os.listdir(directory):
        person_dir = os.path.join(directory, name)
        if os.path.isdir(person_dir):
            for file in os.listdir(person_dir):
                path = os.path.join(person_dir, file)
                image = face_recognition.load_image_file(path)
                enc = face_recognition.face_encodings(image)
                if enc:
                    known_faces.append(enc[0])
                    known_names.append(name)

load_faces(known_faces_dir1)
#load_faces(known_faces_dir2)
print("[INFO] Faces Loaded:", len(known_faces))

last_reload = time.time()
RELOAD_INTERVAL = 30  # Reload every 30 seconds

video = VideoStream(0)
frame_count = 0
last_seen = ""
last_crowd_update = time.time()

trackers = []
tracker_names = []

while True:
    ret, frame = video.read()
    if not ret:
        continue

    frame_count += 1
    
    # Check if we need to reload faces with new missing persons
    if time.time() - last_reload > RELOAD_INTERVAL and hasNewFaces():
        known_faces.clear()
        known_names.clear()
        load_faces(known_faces_dir1)
        print("[INFO] Faces reloaded:", len(known_faces))
        last_reload = time.time()

    if frame_count % DETECT_EVERY_N_FRAMES == 0:
        trackers.clear()
        tracker_names.clear()

        small = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)
        rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)

        locations = face_recognition.face_locations(
            rgb,
            number_of_times_to_upsample=1,
            model=MODEL
        )

        encodings = face_recognition.face_encodings(rgb, locations)

        for (top, right, bottom, left), encoding in zip(locations, encodings):
            name = "Unknown"
            matches = face_recognition.compare_faces(
                known_faces, encoding, TOLERANCE
            )

            if True in matches:
                name = known_names[matches.index(True)]
                if last_seen != name:
                    dataAdd(name)
                    sendEmail(name)
                    last_seen = name

            top *= 2
            right *= 2
            bottom *= 2
            left *= 2

            w = right - left
            h = bottom - top

            tracker = cv2.legacy.TrackerCSRT_create()

            tracker.init(frame, (left, top, w, h))

            trackers.append(tracker)
            tracker_names.append(name)

    people_count = 0
    for tracker, name in zip(trackers, tracker_names):
        success, box = tracker.update(frame)
        if success:
            people_count += 1
            x, y, w, h = map(int, box)
            cv2.rectangle(frame, (x, y), (x + w, y + h),
                          (0, 255, 0), 2)
            cv2.putText(frame, name, (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                        (0, 255, 0), 2)

    if people_count < 3:
        crowd_level = "LOW"
        color = (0, 255, 0)
    elif people_count <= 5:
        crowd_level = "MEDIUM"
        color = (0, 255, 255)
    else:
        crowd_level = "HIGH"
        color = (0, 0, 255)

    if time.time() - last_crowd_update > 5:
        updateCrowdStatus(people_count, crowd_level)
        last_crowd_update = time.time()

    cv2.putText(frame, f"People: {people_count}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv2.putText(frame, f"Level: {crowd_level}", (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    cv2.imshow("Camera", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video.stop()
cv2.destroyAllWindows()
print("[INFO] Cleanup done")